# Qwen2.5-1.5B QLoRA - CSRC RAG Fine-tuning (Colab / Kaggle)

团队成员 · Deeplearning_project-CSRC_Rag · Track B

Runs on T4 (Colab free) or P100 (Kaggle). Expected wall-clock: 3-5 hours for 5.5k samples x 3 epochs.

> **Hardware note — this notebook targets Colab T4 / Kaggle P100 with bf16 compute dtype kept as-is.**
> On the developer's local box (Windows + RTX 2060 SUPER, **Turing** architecture, 8 GB VRAM), bf16 is **unsupported** by the hardware.
> For local runs use `scripts/train_qlora_qwen.py` together with `configs/qlora_config.json`, which is already patched for fp16:
> - `bnb_4bit_compute_dtype=float16`
> - `fp16=True`, `bf16=False`
> - `per_device_train_batch_size=2`, `gradient_accumulation_steps=8` (keeps effective batch=16)

**Steps**
1. Mount Drive / attach Dataset
2. Install pinned deps
3. Load config + data
4. Train QLoRA
5. Save adapter back to Drive/Kaggle output

## 0. Environment detection

In [ ]:
import os, sys, pathlib
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/input')
print('colab=', IS_COLAB, 'kaggle=', IS_KAGGLE)

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = pathlib.Path('/content/drive/MyDrive/csrc_rag')
elif IS_KAGGLE:
    WORK = pathlib.Path('/kaggle/input/csrc-qa-train')
else:
    WORK = pathlib.Path('.')

OUT = pathlib.Path('/content/outputs/qwen_lora_csrc' if IS_COLAB else '/kaggle/working/qwen_lora_csrc')
OUT.mkdir(parents=True, exist_ok=True)
print('WORK=', WORK, 'OUT=', OUT)

## 1. Install pinned dependencies

Pinned versions proven to work with T4 + Qwen2.5 4-bit. Do NOT upgrade blindly.

In [ ]:
%pip install -q \
    transformers==4.44.2 \
    peft==0.13.2 \
    bitsandbytes==0.43.3 \
    accelerate==0.33.0 \
    datasets==2.21.0 \
    trl==0.9.6 \
    sentencepiece

## 2. HuggingFace mirror (China)

In [ ]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

## 3. Load config + data

In [ ]:
import json
CFG = json.loads((WORK/'configs'/'qlora_config.json').read_text(encoding='utf-8'))
print('base_model =', CFG['meta']['base_model'])
print('r =', CFG['lora']['r'], 'alpha =', CFG['lora']['lora_alpha'])

train_path = WORK / 'data' / 'train' / 'rag_qa_train.jsonl'
val_path = WORK / 'data' / 'train' / 'rag_qa_val.jsonl'

def read_jsonl(p):
    with open(p, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = read_jsonl(train_path)
val_data = read_jsonl(val_path)
print('train=', len(train_data), 'val=', len(val_data))

## 4. Load base model in 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL = CFG['meta']['base_model']

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(**{k: v for k, v in CFG['lora'].items() if k != 'task_type'}, task_type='CAUSAL_LM')
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 5. Tokenize

In [ ]:
from datasets import Dataset
MAX_LEN = CFG['training']['max_seq_length']

def encode(row):
    text = tok.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)
    enc = tok(text, max_length=MAX_LEN, truncation=True, padding=False)
    enc['labels'] = enc['input_ids'].copy()
    return enc

train_ds = Dataset.from_list(train_data).map(encode, remove_columns=['messages'])
val_ds = Dataset.from_list(val_data).map(encode, remove_columns=['messages']) if val_data else None
print('train_ds=', train_ds, '\nval_ds=', val_ds)

## 6. Trainer

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling, EarlyStoppingCallback
tr = CFG['training']

args = TrainingArguments(
    output_dir=str(OUT),
    per_device_train_batch_size=tr['per_device_train_batch_size'],
    per_device_eval_batch_size=tr['per_device_eval_batch_size'],
    gradient_accumulation_steps=tr['gradient_accumulation_steps'],
    learning_rate=tr['learning_rate'],
    num_train_epochs=tr['num_train_epochs'],
    warmup_ratio=tr['warmup_ratio'],
    lr_scheduler_type=tr['lr_scheduler_type'],
    weight_decay=tr['weight_decay'],
    optim=tr['optim'],
    gradient_checkpointing=tr['gradient_checkpointing'],
    logging_steps=tr['logging_steps'],
    eval_strategy='steps' if val_ds else 'no',
    eval_steps=tr['eval_steps'],
    save_strategy='steps',
    save_steps=tr['save_steps'],
    save_total_limit=tr['save_total_limit'],
    load_best_model_at_end=bool(val_ds),
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to=['tensorboard'],
    seed=tr['seed'],
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=CFG['early_stopping']['patience'])] if val_ds else []

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
    callbacks=callbacks,
)
trainer.train()
trainer.save_model(str(OUT))
tok.save_pretrained(str(OUT))

## 7. Back up adapter to Drive / Kaggle output

In [ ]:
import shutil
if IS_COLAB:
    dst = '/content/drive/MyDrive/csrc_rag/artifacts/models/qwen_lora_csrc'
    shutil.copytree(str(OUT), dst, dirs_exist_ok=True)
    print('copied to', dst)
elif IS_KAGGLE:
    print('download', str(OUT), 'from Kaggle Notebook Output panel')

## 8. Smoke-test inference

In [ ]:
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True)
infer = PeftModel.from_pretrained(base, str(OUT))
infer.eval()

messages = [
    {'role': 'system', 'content': CFG.get('data', {}).get('system_prompt', '你是证监会处罚案例助手。')},
    {'role': 'user', 'content': '内幕交易历史上通常怎么处罚？'},
]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(text, return_tensors='pt').to(infer.device)
out = infer.generate(**inputs, max_new_tokens=256, temperature=0.2, top_p=0.9, pad_token_id=tok.pad_token_id)
print(tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))